# Phase 8 Step 8.7 — review without tuning

Run every cell once, from top to bottom. In Google Colab, Cell 3 asks you to upload one small ZIP named `phase8_step_8_7_inputs.zip`. The last cell downloads the review results. This notebook does not fit, tune, or select a model.

In [ ]:
from pathlib import Path
import os, subprocess

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/frager098/HalalifyAI.git'
REPO_REF = 'feature/model-evaluation'
if IN_COLAB:
    REPO_ROOT = Path('/content/HalalifyAI')
    if not (REPO_ROOT / 'AGENTS.md').is_file():
        subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    start = Path.cwd().resolve()
    REPO_ROOT = next((p for p in [start, *start.parents] if (p / 'AGENTS.md').is_file() and (p / 'src').is_dir() and (p / 'configs').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Could not locate the HalalifyAI repository root.')

os.chdir(REPO_ROOT)
print('Environment:', 'Google Colab browser' if IN_COLAB else 'Local Python')
print('Repository root:', REPO_ROOT)
print('Git ref:', REPO_REF)

In [ ]:
# Colab input upload. A browser Choose files dialog appears only when inputs are missing.
import json, zipfile
from pathlib import PurePosixPath

required_code = [
    Path('configs/phase8_review_v1.json'),
    Path('src/model_training/phase8/review.py'),
]
missing_code = [str(path) for path in required_code if not path.is_file()]
if missing_code:
    raise FileNotFoundError('The cloned branch lacks Step 8.7 code. Commit and push feature/model-evaluation first. Missing: ' + ', '.join(missing_code))

review_config = json.loads(Path('configs/phase8_review_v1.json').read_text(encoding='utf-8'))
required_paths = [spec['path'] for spec in review_config['required_inputs'].values()]
missing_inputs = [path for path in required_paths if not Path(path).is_file()]
if IN_COLAB and missing_inputs:
    from google.colab import files  # type: ignore
    print('Upload phase8_step_8_7_inputs.zip now. It must contain these repository-relative files:')
    for path in required_paths:
        print(' -', path)
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise FileNotFoundError('Expected one ZIP upload; received: ' + ', '.join(uploaded.keys()))
    print('Received ZIP:', zip_names[0])
    with zipfile.ZipFile(__import__('io').BytesIO(uploaded[zip_names[0]])) as archive:
        names = set(archive.namelist())
        missing_in_zip = [path for path in required_paths if path not in names]
        if missing_in_zip:
            raise FileNotFoundError('The ZIP lacks: ' + ', '.join(missing_in_zip))
        unexpected_files = [name for name in names if not name.endswith('/') and name not in required_paths]
        if unexpected_files:
            raise ValueError('The ZIP contains unexpected files: ' + ', '.join(unexpected_files))
        for relative in required_paths:
            safe = PurePosixPath(relative)
            if safe.is_absolute() or '..' in safe.parts:
                raise ValueError(f'Unsafe ZIP path: {relative}')
            destination = REPO_ROOT.joinpath(*safe.parts)
            if destination.exists():
                raise FileExistsError(f'Refusing to overwrite frozen evidence: {destination}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(relative))

missing_after = [path for path in required_paths if not Path(path).is_file()]
if missing_after:
    raise FileNotFoundError('Required inputs are missing: ' + ', '.join(missing_after))
print('All Step 8.7 input files are present.')

In [ ]:
from src.model_training.phase8.review import validate_review_inputs
validation = validate_review_inputs(REPO_ROOT)
print('Preflight and SHA-256 validation passed.')
for name, identity in validation['identities'].items():
    print(f"{name}: {identity['sha256']}")

In [ ]:
from src.model_training.phase8.review import run_step_8_7_review
review = run_step_8_7_review(REPO_ROOT)
print('Step 8.7 review created without fitting or tuning.')
print('JSON:', review_config['outputs']['json'])
print('Markdown:', review_config['outputs']['markdown'])

In [ ]:
print('RETURN REVIEW')
print(review['return_review']['finding'])
print('Reserved Spearman:', review['return_review']['reserved_mean_daily_spearman'])
print('Phase 7 fold mean:', review['return_review']['development_mean_daily_spearman'])
print()
print('RISK REVIEW')
print(review['risk_review']['finding'])
print('Reserved macro F1:', review['risk_review']['reserved_macro_f1'])
print('Phase 7 fold mean:', review['risk_review']['development_mean_macro_f1'])
print()
print('CONCLUSION')
print(review['conclusion']['research_interpretation'])

## Download the evidence

Run the final cell. In Colab it downloads `phase8_step_8_7_results.zip`. Keep that ZIP and copy its two files back into the same repository-relative report directory.

In [ ]:
# Final cell: package and download only the Step 8.7 outputs.
import tempfile, zipfile

output_paths = [Path(review_config['outputs']['json']), Path(review_config['outputs']['markdown'])]
for path in output_paths:
    if not path.is_file():
        raise FileNotFoundError(f'Missing review output: {path}')
archive_path = Path('/content/phase8_step_8_7_results.zip') if IN_COLAB else Path(tempfile.gettempdir()) / 'phase8_step_8_7_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=path.as_posix())
print('Created:', archive_path)
if IN_COLAB:
    from google.colab import files  # type: ignore
    files.download(str(archive_path))
else:
    print('Local run: copy the ZIP from the path above if needed.')